# Bloc 3.2 — KPI design

**Decision problem:** which metrics tell us whether attention is stable, seasonal, or shifting?

Output: KPI definitions and snapshot.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
SNAPSHOT = "iot_FR_today_5-y_chatgpt_iphone_meteo.csv"
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "data" / "snapshots" / SNAPSHOT).exists():
        ROOT = candidate
        break
OUT = ROOT / "outputs"
for d in [OUT / "bloc1", OUT / "bloc2", OUT / "bloc3", OUT / "final_product"]: d.mkdir(parents=True, exist_ok=True)
DATA = ROOT / "data" / "snapshots"
def load_clean_long():
    p=OUT/"bloc1"/"clean_trends_long.csv"
    if p.exists(): return pd.read_csv(p, parse_dates=["date"])
    raw=pd.read_csv(DATA/"iot_FR_today_5-y_chatgpt_iphone_meteo.csv", parse_dates=["date"])
    return raw.melt(id_vars="date", var_name="signal", value_name="interest")
def load_clean_wide():
    p=OUT/"bloc1"/"clean_trends_wide.csv"
    if p.exists(): return pd.read_csv(p, parse_dates=["date"])
    raw=pd.read_csv(DATA/"iot_FR_today_5-y_chatgpt_iphone_meteo.csv", parse_dates=["date"])
    return raw.sort_values("date")

In [ ]:
df=load_clean_long()
latest=df.sort_values("date").groupby("signal").tail(1).set_index("signal")["interest"]
median=df.groupby("signal")["interest"].median()
kpi=pd.DataFrame({"latest_interest":latest,"median_interest":median})
kpi["latest_vs_median"]=kpi["latest_interest"]-kpi["median_interest"]
defs=pd.DataFrame([{"kpi":"latest_interest","meaning":"current attention index"},{"kpi":"latest_vs_median","meaning":"simple shift indicator"}])
defs.to_csv(OUT/"bloc3"/"kpi_definitions.csv", index=False)
kpi.round(2).to_csv(OUT/"bloc3"/"kpi_snapshot.csv")
kpi.round(2)

## Exercise

Add one KPI that captures volatility.

## Conclusion

Good KPIs make the decision easier, not the report longer.